In [ ]:
from google.colab import files

archivos = files.upload()

nombre_archivo = next(iter(archivos))
print("Archivo cargado:", nombre_archivo)

del archivos


Saving fallecidos_sinadef.csv to fallecidos_sinadef (1).csv
Archivo cargado: fallecidos_sinadef (1).csv


In [ ]:
import pandas as pd

# Probar codificaciones sin alterar el archivo original.
for codificacion in ["utf-8-sig", "cp1252", "latin1"]:
    try:
        muestra = pd.read_csv(
            nombre_archivo,
            sep="|",
            encoding=codificacion,
            dtype=str,
            keep_default_na=False,
            nrows=1000
        )
        break
    except UnicodeDecodeError:
        continue

muestra.columns = muestra.columns.str.strip()

print("Codificación usada para la muestra:", codificacion)
print("Filas de la muestra:", len(muestra))
print("Número de columnas:", len(muestra.columns))

print("\nCOLUMNAS:")
print(muestra.columns.tolist())

print("\nEJEMPLOS DE FECHA Y EDAD:")
display(muestra[["FECHA", "EDAD", "TIEMPO EDAD"]].head(10))

print("\nUNIDADES DE EDAD EN ESTA MUESTRA:")
print(muestra["TIEMPO EDAD"].value_counts())

Codificación usada para la muestra: utf-8-sig
Filas de la muestra: 1000
Número de columnas: 32

COLUMNAS:
['Nº', 'TIPO SEGURO', 'SEXO', 'EDAD', 'TIEMPO EDAD', 'ESTADO CIVIL', 'NIVEL DE INSTRUCCIÓN', 'ETNIA', 'COD# UBIGEO DOMICILIO', 'PAIS DOMICILIO', 'DEPARTAMENTO DOMICILIO', 'PROVINCIA DOMICILIO', 'DISTRITO DOMICILIO', 'FECHA', 'AÑO', 'MES', 'TIPO LUGAR', 'INSTITUCION', 'MUERTE VIOLENTA', 'NECROPSIA', 'DEBIDO A (CAUSA A)', 'CAUSA A (CIE-X)', 'DEBIDO A (CAUSA B)', 'CAUSA B (CIE-X)', 'DEBIDO A (CAUSA C)', 'CAUSA C (CIE-X)', 'DEBIDO A (CAUSA D)', 'CAUSA D (CIE-X)', 'DEBIDO A (CAUSA E)', 'CAUSA E (CIE-X)', 'DEBIDO A (CAUSA F)', 'CAUSA F (CIE-X)']

EJEMPLOS DE FECHA Y EDAD:


,FECHA,EDAD,TIEMPO EDAD
0,2019-05-25,54,AÑOS
1,2024-01-31,41,AÑOS
2,2024-02-01,84,AÑOS
3,2018-12-25,82,AÑOS
4,2017-11-03,54,AÑOS
5,2024-01-31,92,AÑOS
6,2024-02-01,70,AÑOS
7,2018-07-15,85,AÑOS
8,2018-11-13,60,AÑOS
9,2019-10-09,73,AÑOS



UNIDADES DE EDAD EN ESTA MUESTRA:
TIEMPO EDAD
AÑOS            986
DIAS              8
MESES             5
SIN REGISTRO      1
Name: count, dtype: int64


In [ ]:
import pandas as pd

salida = "sinadef_2022_2024.csv"
revision = "sinadef_fechas_por_revisar.csv"

total = conservados = fuera_periodo = sin_fecha_valida = 0
por_anio = {2022: 0, 2023: 0, 2024: 0}

with open(salida, "w", encoding="utf-8-sig", newline="") as f_salida, \
     open(revision, "w", encoding="utf-8-sig", newline="") as f_revision:

    # Escribir encabezados incluso si no hubiera registros.
    muestra.iloc[:0].to_csv(f_salida, sep="|", index=False)
    muestra.iloc[:0].to_csv(f_revision, sep="|", index=False)

    lector = pd.read_csv(
        nombre_archivo,
        sep="|",
        encoding=codificacion,
        dtype=str,
        keep_default_na=False,
        chunksize=100_000,
        on_bad_lines="error"
    )

    for bloque in lector:
        bloque.columns = bloque.columns.str.strip()

        fechas = pd.to_datetime(
            bloque["FECHA"].str.strip(),
            format="%Y-%m-%d",
            errors="coerce"
        )

        dentro = (
            (fechas >= pd.Timestamp("2022-01-01")) &
            (fechas < pd.Timestamp("2025-01-01"))
        )
        revisar = fechas.isna()

        bloque.loc[dentro].to_csv(
            f_salida, sep="|", index=False, header=False
        )
        bloque.loc[revisar].to_csv(
            f_revision, sep="|", index=False, header=False
        )

        total += len(bloque)
        conservados += int(dentro.sum())
        sin_fecha_valida += int(revisar.sum())
        fuera_periodo += int((fechas.notna() & ~dentro).sum())

        conteos = fechas.loc[dentro].dt.year.value_counts()
        for anio in por_anio:
            por_anio[anio] += int(conteos.get(anio, 0))

        print(f"Registros procesados: {total:,}")

assert total == conservados + fuera_periodo + sin_fecha_valida

resumen = pd.DataFrame({
    "Concepto": [
        "Registros leídos",
        "Conservados 2022–2024",
        "Fuera del periodo",
        "Sin fecha válida",
        "Conservados 2022",
        "Conservados 2023",
        "Conservados 2024"
    ],
    "Cantidad": [
        total, conservados, fuera_periodo, sin_fecha_valida,
        por_anio[2022], por_anio[2023], por_anio[2024]
    ]
})

resumen.to_csv(
    "resumen_filtro_temporal.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFiltro temporal terminado:")
display(resumen)

Registros procesados: 100,000
Registros procesados: 200,000
Registros procesados: 300,000
Registros procesados: 400,000
Registros procesados: 500,000
Registros procesados: 600,000
Registros procesados: 700,000
Registros procesados: 800,000
Registros procesados: 900,000
Registros procesados: 1,000,000
Registros procesados: 1,100,000
Registros procesados: 1,110,168

Filtro temporal terminado:


,Concepto,Cantidad
0,Registros leídos,1110168
1,Conservados 2022–2024,308932
2,Fuera del periodo,801236
3,Sin fecha válida,0
4,Conservados 2022,158577
5,Conservados 2023,110664
6,Conservados 2024,39691


In [ ]:
# Leer únicamente la columna de fechas del archivo filtrado.
datos_fechas = pd.read_csv(
    "sinadef_2022_2024.csv",
    sep="|",
    encoding="utf-8-sig",
    usecols=["FECHA"],
    dtype=str
)

fechas = pd.to_datetime(
    datos_fechas["FECHA"].str.strip(),
    format="%Y-%m-%d",
    errors="raise"
)

# Primera y última fecha con registros en cada año.
cobertura = fechas.groupby(fechas.dt.year).agg(
    ["min", "max", "size"]
)
cobertura.columns = [
    "Primera fecha", "Última fecha", "Registros"
]
cobertura.index.name = "Año"

print("FECHAS PRESENTES POR AÑO:")
display(cobertura)

# Conteos por mes; un cero significa que no hay registros
# de ese mes en este archivo, sin determinar todavía la causa.
mensual = pd.crosstab(fechas.dt.year, fechas.dt.month)
mensual = mensual.reindex(
    index=[2022, 2023, 2024],
    columns=range(1, 13),
    fill_value=0
)
mensual.index.name = "Año"
mensual.columns.name = "Mes"

print("\nREGISTROS POR MES:")
display(mensual)

FECHAS PRESENTES POR AÑO:


,Primera fecha,Última fecha,Registros
Año,,,
2022,2022-01-01,2022-12-31,158577
2023,2023-01-01,2023-12-31,110664
2024,2024-01-01,2024-05-05,39691



REGISTROS POR MES:


Mes,1,2,3,4,5,6,7,8,9,10,11,12
Año,,,,,,,,,,,,
2022,18261,15815,13825,12498,13143,13303,14600,15177,13455,9813,8772,9915
2023,9156,8355,9787,10031,9935,9165,9210,9232,8947,9378,8773,8695
2024,9724,9275,10147,9336,1209,0,0,0,0,0,0,0


In [ ]:
import zipfile
from google.colab import files

cobertura.to_csv("cobertura_por_anio.csv", encoding="utf-8-sig")
mensual.to_csv("registros_por_mes.csv", encoding="utf-8-sig")

with zipfile.ZipFile(
    "SINADEF_revision_temporal.zip",
    "w",
    compression=zipfile.ZIP_DEFLATED
) as paquete:
    for archivo in [
        "sinadef_2022_2024.csv",
        "sinadef_fechas_por_revisar.csv",
        "resumen_filtro_temporal.csv",
        "cobertura_por_anio.csv",
        "registros_por_mes.csv"
    ]:
        paquete.write(archivo)

files.download("SINADEF_revision_temporal.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

geografia = pd.read_csv(
    "sinadef_2022_2024.csv",
    sep="|",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
    usecols=[
        "DEPARTAMENTO DOMICILIO",
        "PROVINCIA DOMICILIO"
    ]
)

print("DEPARTAMENTOS DE DOMICILIO:")
display(
    geografia["DEPARTAMENTO DOMICILIO"]
    .value_counts()
    .rename_axis("Departamento")
    .reset_index(name="Registros")
)

# Comparar nombres ignorando espacios externos y mayúsculas.
es_lima = (
    geografia["DEPARTAMENTO DOMICILIO"]
    .str.strip()
    .str.upper()
    .eq("LIMA")
)

print("\nPROVINCIAS DEL DEPARTAMENTO LIMA:")
display(
    geografia.loc[es_lima, "PROVINCIA DOMICILIO"]
    .value_counts()
    .rename_axis("Provincia")
    .reset_index(name="Registros")
)

DEPARTAMENTOS DE DOMICILIO:


,Departamento,Registros
0,LIMA,111517
1,LA LIBERTAD,20079
2,AREQUIPA,15584
3,PUNO,14908
4,PIURA,14552
5,ANCASH,14228
6,JUNIN,13214
7,ICA,11524
8,CUSCO,11246
9,CALLAO,11202



PROVINCIAS DEL DEPARTAMENTO LIMA:


,Provincia,Registros
0,LIMA,102832
1,CAÑETE,2453
2,HUARAL,1776
3,HUAURA,1647
4,HUAROCHIRI,1136
5,BARRANCA,999
6,YAUYOS,308
7,CANTA,190
8,OYON,120
9,CAJATAMBO,43


In [ ]:
import pandas as pd

archivo_entrada = "sinadef_2022_2024.csv"
archivo_lima = "sinadef_lima_provincia_2022_2024.csv"

total_entrada = 0
total_departamento = 0
total_provincia = 0
primer_bloque = True

with open(
    archivo_lima, "w", encoding="utf-8-sig", newline=""
) as destino:

    for bloque in pd.read_csv(
        archivo_entrada,
        sep="|",
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False,
        chunksize=50_000,
        on_bad_lines="error"
    ):
        departamento = (
            bloque["DEPARTAMENTO DOMICILIO"]
            .str.strip().str.upper()
        )
        provincia = (
            bloque["PROVINCIA DOMICILIO"]
            .str.strip().str.upper()
        )

        es_departamento_lima = departamento.eq("LIMA")
        es_provincia_lima = (
            es_departamento_lima & provincia.eq("LIMA")
        )

        seleccion = bloque.loc[es_provincia_lima]

        seleccion.to_csv(
            destino,
            sep="|",
            index=False,
            header=primer_bloque
        )
        primer_bloque = False

        total_entrada += len(bloque)
        total_departamento += int(es_departamento_lima.sum())
        total_provincia += len(seleccion)

resumen_geografico = pd.DataFrame({
    "Etapa": [
        "Base temporal de entrada",
        "Departamento Lima",
        "Departamento Lima y provincia Lima"
    ],
    "Registros": [
        total_entrada,
        total_departamento,
        total_provincia
    ]
})

resumen_geografico.to_csv(
    "resumen_filtro_geografico.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_geografico)
print("Archivo generado:", archivo_lima)

,Etapa,Registros
0,Base temporal de entrada,308932
1,Departamento Lima,111517
2,Departamento Lima y provincia Lima,102832


Archivo generado: sinadef_lima_provincia_2022_2024.csv


In [ ]:
from google.colab import files

files.download("sinadef_lima_provincia_2022_2024.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

edades = pd.read_csv(
    "sinadef_lima_provincia_2022_2024.csv",
    sep="|",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
    usecols=["EDAD", "TIEMPO EDAD"]
)

unidad = (
    edades["TIEMPO EDAD"]
    .str.strip()
    .str.upper()
    .str.replace("Ñ", "N", regex=False)
)

edad_numerica = pd.to_numeric(
    edades["EDAD"].str.strip(),
    errors="coerce"
)

en_anios = unidad.eq("ANOS")

print("UNIDADES DE EDAD REGISTRADAS:")
display(
    edades["TIEMPO EDAD"]
    .value_counts()
    .rename_axis("Unidad")
    .reset_index(name="Registros")
)

resumen_edades = pd.DataFrame({
    "Comprobación": [
        "Total de registros",
        "Edad expresada en años",
        "Exactamente 60 años",
        "Más de 60 años",
        "60 años o más",
        "Edad vacía o no numérica"
    ],
    "Registros": [
        len(edades),
        int(en_anios.sum()),
        int((en_anios & edad_numerica.eq(60)).sum()),
        int((en_anios & edad_numerica.gt(60)).sum()),
        int((en_anios & edad_numerica.ge(60)).sum()),
        int(edad_numerica.isna().sum())
    ]
})

print("\nCOMPROBACIONES DE EDAD:")
display(resumen_edades)

print("\nMAYORES VALORES REGISTRADOS EN AÑOS:")
display(
    edad_numerica.loc[en_anios]
    .dropna()
    .drop_duplicates()
    .sort_values(ascending=False)
    .head(15)
    .to_frame(name="Edad")
)

UNIDADES DE EDAD REGISTRADAS:


,Unidad,Registros
0,AÑOS,100978
1,DIAS,875
2,MESES,564
3,HORAS,255
4,MINUTOS,159
5,SIN REGISTRO,1



COMPROBACIONES DE EDAD:


,Comprobación,Registros
0,Total de registros,102832
1,Edad expresada en años,100978
2,Exactamente 60 años,1146
3,Más de 60 años,77151
4,60 años o más,78297
5,Edad vacía o no numérica,1



MAYORES VALORES REGISTRADOS EN AÑOS:


,Edad
61550,113.0
18268,111.0
15653,109.0
11607,108.0
12296,107.0
270,106.0
1658,105.0
1034,104.0
2078,103.0
87,102.0


In [ ]:
import pandas as pd

entrada = "sinadef_lima_provincia_2022_2024.csv"
salida = "sinadef_lima_60mas_2022_2024.csv"

# Leer todas las columnas como texto para conservar sus valores.
datos = pd.read_csv(
    entrada,
    sep="|",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

# Preparar variables auxiliares sin modificar las columnas originales.
unidad = (
    datos["TIEMPO EDAD"]
    .str.strip()
    .str.upper()
    .str.replace("Ñ", "N", regex=False)
)

edad = pd.to_numeric(
    datos["EDAD"].str.strip(),
    errors="coerce"
)

# Incluir únicamente edades de 60 años a más.
seleccion = unidad.eq("ANOS") & edad.ge(60)
filtrados = datos.loc[seleccion].copy()

filtrados.to_csv(
    salida,
    sep="|",
    encoding="utf-8-sig",
    index=False
)

# Guardar un resumen para documentar el filtro.
resumen = pd.DataFrame({
    "Etapa": [
        "Registros de Lima antes del filtro de edad",
        "Incluidos: edad en años y mayor o igual a 60",
        "Excluidos por el criterio de edad o su unidad"
    ],
    "Registros": [
        len(datos),
        len(filtrados),
        len(datos) - len(filtrados)
    ]
})

resumen.to_csv(
    "resumen_filtro_edad.csv",
    encoding="utf-8-sig",
    index=False
)

display(resumen)
print("Archivo generado:", salida)

,Etapa,Registros
0,Registros de Lima antes del filtro de edad,102832
1,Incluidos: edad en años y mayor o igual a 60,78297
2,Excluidos por el criterio de edad o su unidad,24535


Archivo generado: sinadef_lima_60mas_2022_2024.csv


In [ ]:
from google.colab import files
import zipfile

with zipfile.ZipFile(
    "SINADEF_Lima_60mas.zip", "w", zipfile.ZIP_DEFLATED
) as paquete:
    paquete.write("sinadef_lima_60mas_2022_2024.csv")
    paquete.write("resumen_filtro_edad.csv")

files.download("SINADEF_Lima_60mas.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

ubicacion = pd.read_csv(
    "sinadef_lima_60mas_2022_2024.csv",
    sep="|",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
    usecols=[
        "COD# UBIGEO DOMICILIO",
        "DISTRITO DOMICILIO"
    ]
)

# Limpiar espacios únicamente en esta tabla de revisión.
ubicacion = ubicacion.apply(lambda columna: columna.str.strip())

tabla_distritos = (
    ubicacion
    .groupby(
        ["DISTRITO DOMICILIO", "COD# UBIGEO DOMICILIO"],
        dropna=False
    )
    .size()
    .reset_index(name="REGISTROS")
    .sort_values(
        ["DISTRITO DOMICILIO", "COD# UBIGEO DOMICILIO"]
    )
)

print("DISTRITOS Y CÓDIGOS REGISTRADOS")
display(tabla_distritos)

# Revisar únicamente el formato del UBIGEO.
formato_correcto = ubicacion[
    "COD# UBIGEO DOMICILIO"
].str.fullmatch(r"[0-9]{6}", na=False)

print(
    "Registros con UBIGEO vacío o sin formato de seis dígitos:",
    int((~formato_correcto).sum())
)

if (~formato_correcto).any():
    display(
        ubicacion.loc[~formato_correcto]
        .value_counts()
        .reset_index(name="REGISTROS")
    )

DISTRITOS Y CÓDIGOS REGISTRADOS


,DISTRITO DOMICILIO,COD# UBIGEO DOMICILIO,REGISTROS
0,ANCON,92-33-14-01-02-000,343
1,ATE,92-33-14-01-03-000,3554
2,BARRANCO,92-33-14-01-25-000,674
3,BREÑA,92-33-14-01-04-000,1557
4,CARABAYLLO,92-33-14-01-05-000,1739
5,CHACLACAYO,92-33-14-01-07-000,525
6,CHORRILLOS,92-33-14-01-08-000,2580
7,CIENEGUILLA,92-33-14-01-39-000,222
8,COMAS,92-33-14-01-06-000,4369
9,EL AGUSTINO,92-33-14-01-35-000,1689


Registros con UBIGEO vacío o sin formato de seis dígitos: 78297


,COD# UBIGEO DOMICILIO,DISTRITO DOMICILIO,REGISTROS
0,92-33-14-01-37-000,SAN JUAN DE LURIGANCHO,6207
1,92-33-14-01-26-000,SAN MARTIN DE PORRES,5458
2,92-33-14-01-06-000,COMAS,4369
3,92-33-14-01-30-000,SANTIAGO DE SURCO,4028
4,92-33-14-01-01-000,LIMA,3983
5,92-33-14-01-03-000,ATE,3554
6,92-33-14-01-36-000,SAN JUAN DE MIRAFLORES,3246
7,92-33-14-01-32-000,VILLA MARIA DEL TRIUNFO,2990
8,92-33-14-01-41-000,VILLA EL SALVADOR,2625
9,92-33-14-01-09-000,LA VICTORIA,2613


In [ ]:
# Utiliza la tabla generada en la celda anterior.
resumen_distritos = (
    tabla_distritos
    .groupby("DISTRITO DOMICILIO", as_index=False)["REGISTROS"]
    .sum()
    .sort_values("DISTRITO DOMICILIO")
)

resumen_distritos.to_csv(
    "resumen_registros_por_distrito.csv",
    encoding="utf-8-sig",
    index=False
)

tabla_distritos.to_csv(
    "revision_distritos_codigos_originales.csv",
    encoding="utf-8-sig",
    index=False
)

sin_distrito = resumen_distritos["DISTRITO DOMICILIO"].isin(
    ["", "SIN REGISTRO"]
)

print("Total de registros:", resumen_distritos["REGISTROS"].sum())
print(
    "Con nombre de distrito:",
    resumen_distritos.loc[~sin_distrito, "REGISTROS"].sum()
)
print(
    "Sin distrito identificado:",
    resumen_distritos.loc[sin_distrito, "REGISTROS"].sum()
)
print("Tablas de revisión guardadas.")

Total de registros: 78297
Con nombre de distrito: 78286
Sin distrito identificado: 11
Tablas de revisión guardadas.


In [ ]:
import pandas as pd
import unicodedata

datos = pd.read_csv(
    "sinadef_lima_60mas_2022_2024.csv",
    sep="|",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.upper().split())

# Esta correspondencia se aplica exclusivamente a Lima, provincia Lima.
dentro_de_lima = (
    datos["DEPARTAMENTO DOMICILIO"].map(normalizar).eq("LIMA")
    & datos["PROVINCIA DOMICILIO"].map(normalizar).eq("LIMA")
)

if not dentro_de_lima.all():
    raise ValueError("Hay registros fuera de Lima–Lima. Revisar antes de continuar.")

ubigeos = {
    "LIMA": "150101",
    "ANCON": "150102",
    "ATE": "150103",
    "BARRANCO": "150104",
    "BRENA": "150105",
    "CARABAYLLO": "150106",
    "CHACLACAYO": "150107",
    "CHORRILLOS": "150108",
    "CIENEGUILLA": "150109",
    "COMAS": "150110",
    "EL AGUSTINO": "150111",
    "INDEPENDENCIA": "150112",
    "JESUS MARIA": "150113",
    "LA MOLINA": "150114",
    "LA VICTORIA": "150115",
    "LINCE": "150116",
    "LOS OLIVOS": "150117",
    "LURIGANCHO": "150118",
    "LURIN": "150119",
    "MAGDALENA DEL MAR": "150120",
    "PUEBLO LIBRE": "150121",
    "MIRAFLORES": "150122",
    "PACHACAMAC": "150123",
    "PUCUSANA": "150124",
    "PUENTE PIEDRA": "150125",
    "PUNTA HERMOSA": "150126",
    "PUNTA NEGRA": "150127",
    "RIMAC": "150128",
    "SAN BARTOLO": "150129",
    "SAN BORJA": "150130",
    "SAN ISIDRO": "150131",
    "SAN JUAN DE LURIGANCHO": "150132",
    "SAN JUAN DE MIRAFLORES": "150133",
    "SAN LUIS": "150134",
    "SAN MARTIN DE PORRES": "150135",
    "SAN MIGUEL": "150136",
    "SANTA ANITA": "150137",
    "SANTA MARIA DEL MAR": "150138",
    "SANTA ROSA": "150139",
    "SANTIAGO DE SURCO": "150140",
    "SURQUILLO": "150141",
    "VILLA EL SALVADOR": "150142",
    "VILLA MARIA DEL TRIUNFO": "150143"
}

# Añadir el código sin reemplazar ninguna columna original.
distritos_normalizados = datos["DISTRITO DOMICILIO"].map(normalizar)
datos["UBIGEO_INEI"] = distritos_normalizados.map(ubigeos).fillna("")

# Documentar cómo se asignó cada código.
datos["ESTADO_UBIGEO"] = "ASIGNADO_POR_NOMBRE_DISTRITO"
datos.loc[datos["UBIGEO_INEI"].eq(""), "ESTADO_UBIGEO"] = "PENDIENTE"
datos.loc[
    distritos_normalizados.isin(["", "SIN REGISTRO"]),
    "ESTADO_UBIGEO"
] = "SIN_DISTRITO_IDENTIFICADO"

# Guardar una nueva versión con todas las filas y columnas.
salida = "sinadef_lima_60mas_2022_2024_ubigeo.csv"
datos.to_csv(
    salida,
    sep="|",
    encoding="utf-8-sig",
    index=False
)

# Guardar la correspondencia para poder revisar el procedimiento.
equivalencias = (
    datos.groupby(
        [
            "DISTRITO DOMICILIO",
            "COD# UBIGEO DOMICILIO",
            "UBIGEO_INEI",
            "ESTADO_UBIGEO"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="REGISTROS")
)

equivalencias["FUENTE_CATALOGO"] = (
    "https://www.sunat.gob.pe/legislacion/"
    "superin/2023/anexo-000103-2023.pdf"
)

equivalencias.to_csv(
    "equivalencias_ubigeo_sinadef_lima.csv",
    encoding="utf-8-sig",
    index=False
)

print("Total conservado:", len(datos))
print("Con UBIGEO asignado:", datos["UBIGEO_INEI"].ne("").sum())
print("Sin UBIGEO asignado:", datos["UBIGEO_INEI"].eq("").sum())
print("Distritos con código asignado:",
      datos.loc[datos["UBIGEO_INEI"].ne(""), "UBIGEO_INEI"].nunique())
print("Archivo generado:", salida)

Total conservado: 78297
Con UBIGEO asignado: 78286
Sin UBIGEO asignado: 11
Distritos con código asignado: 43
Archivo generado: sinadef_lima_60mas_2022_2024_ubigeo.csv


In [ ]:
import pandas as pd

datos = pd.read_csv(
    "sinadef_lima_60mas_2022_2024_ubigeo.csv",
    sep="|",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

# Localizar las columnas de códigos de causas.
columnas_cie = [
    columna for columna in datos.columns
    if "CIE" in columna.upper()
]

if not columnas_cie:
    raise ValueError("No se encontraron columnas CIE. Revisar los encabezados.")

# Preparar una copia para inspección.
codigos = datos[columnas_cie].apply(
    lambda columna: columna.str.strip().str.upper()
)

sin_informacion = [
    "", "SIN REGISTRO", "SIN DATO", "SIN DATOS",
    "NO REGISTRADO", "NULL", "NAN"
]
codigos = codigos.replace(sin_informacion, pd.NA)

resumen_causas = pd.DataFrame({
    "CAMPO": columnas_cie,
    "CON_CONTENIDO": codigos.notna().sum().to_numpy(),
    "SIN_INFORMACION": codigos.isna().sum().to_numpy()
})

print("DISPONIBILIDAD POR CAMPO")
display(resumen_causas)

print(
    "Registros sin información en ninguno de los campos CIE:",
    int(codigos.isna().all(axis=1).sum())
)

# Contar apariciones de cada código en todos los campos.
frecuencias = (
    codigos.stack()
    .value_counts()
    .head(20)
    .rename_axis("CODIGO_REGISTRADO")
    .reset_index(name="MENCIONES")
)

print("\n20 CÓDIGOS MÁS FRECUENTES ENTRE TODOS LOS CAMPOS")
display(frecuencias)

DISPONIBILIDAD POR CAMPO


,CAMPO,CON_CONTENIDO,SIN_INFORMACION
0,CAUSA A (CIE-X),63675,14622
1,CAUSA B (CIE-X),52048,26249
2,CAUSA C (CIE-X),31530,46767
3,CAUSA D (CIE-X),10058,68239
4,CAUSA E (CIE-X),11877,66420
5,CAUSA F (CIE-X),4180,74117


Registros sin información en ninguno de los campos CIE: 14175

20 CÓDIGOS MÁS FRECUENTES ENTRE TODOS LOS CAMPOS


,CODIGO_REGISTRADO,MENCIONES
0,J960,22413
1,J189,10558
2,A419,9897
3,I219,9634
4,J969,9628
5,I10X,7101
6,J159,3894
7,R572,3680
8,J849,3332
9,R579,3306


In [ ]:
# Revisar los campos de descripción de las causas.
columnas_texto = [
    columna for columna in datos.columns
    if columna.strip().upper().startswith("DEBIDO A")
]

if not columnas_texto:
    raise ValueError("No se encontraron las descripciones de las causas.")

textos = datos[columnas_texto].apply(
    lambda columna: columna.str.strip().str.upper()
).replace(sin_informacion, pd.NA)

sin_cie = codigos.isna().all(axis=1)
con_texto = textos.notna().any(axis=1)

revision = pd.DataFrame({
    "ANIO": pd.to_datetime(
        datos["FECHA"],
        format="%Y-%m-%d",
        errors="coerce"
    ).dt.year,
    "SIN_CIE": sin_cie,
    "SIN_CIE_CON_TEXTO": sin_cie & con_texto,
    "SIN_CIE_NI_TEXTO": sin_cie & ~con_texto
})

calidad_por_anio = (
    revision.groupby("ANIO", dropna=False)
    .agg(
        TOTAL=("SIN_CIE", "size"),
        SIN_CIE=("SIN_CIE", "sum"),
        SIN_CIE_CON_TEXTO=("SIN_CIE_CON_TEXTO", "sum"),
        SIN_CIE_NI_TEXTO=("SIN_CIE_NI_TEXTO", "sum")
    )
    .reset_index()
)

calidad_por_anio["PORCENTAJE_SIN_CIE"] = (
    100 * calidad_por_anio["SIN_CIE"] / calidad_por_anio["TOTAL"]
).round(2)

display(calidad_por_anio)

calidad_por_anio.to_csv(
    "calidad_causas_por_anio.csv",
    encoding="utf-8-sig",
    index=False
)

print("Revisión guardada. Se mantienen todos los registros.")

,ANIO,TOTAL,SIN_CIE,SIN_CIE_CON_TEXTO,SIN_CIE_NI_TEXTO,PORCENTAJE_SIN_CIE
0,2022,37148,7605,7605,0,20.47
1,2023,29864,4878,4878,0,16.33
2,2024,11285,1692,1692,0,14.99


Revisión guardada. Se mantienen todos los registros.


In [ ]:
# Revisar las descripciones de los registros sin ningún código CIE.
descripciones = (
    textos.loc[sin_cie]
    .stack()
    .rename_axis(["FILA", "CAMPO"])
    .reset_index(name="DESCRIPCION")
)

# Contar cada descripción una sola vez por registro,
# aunque se repita en varios campos de esa misma fila.
descripciones = descripciones.drop_duplicates(
    subset=["FILA", "DESCRIPCION"]
)

frecuencias_texto = (
    descripciones["DESCRIPCION"]
    .value_counts()
    .rename_axis("DESCRIPCION")
    .reset_index(name="REGISTROS_QUE_LA_MENCIONAN")
)

print("20 DESCRIPCIONES MÁS FRECUENTES EN REGISTROS SIN CIE")
display(frecuencias_texto.head(20))

frecuencias_texto.to_csv(
    "revision_descripciones_sin_cie.csv",
    encoding="utf-8-sig",
    index=False
)

print("Resumen guardado. La base principal permanece intacta.")

20 DESCRIPCIONES MÁS FRECUENTES EN REGISTROS SIN CIE


,DESCRIPCION,REGISTROS_QUE_LA_MENCIONAN
0,FALLA MULTIORGANICA,2027
1,INSUFICIENCIA RESPIRATORIA AGUDA,1849
2,INSUFICIENCIA RESPIRATORIA,1774
3,HIPERTENSION ARTERIAL,1654
4,INFARTO AGUDO DE MIOCARDIO,1142
5,NEUMONIA,934
6,INSUFICIENCIA CARDIACA,895
7,SEPSIS,716
8,SHOCK SEPTICO,713
9,FIBROSIS PULMONAR,696


Resumen guardado. La base principal permanece intacta.


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

preparados = datos.copy()

# Estos indicadores describen disponibilidad, no validación diagnóstica.
preparados["TIENE_CIE_REGISTRADO"] = (
    codigos.notna().any(axis=1).map({True: "SI", False: "NO"})
)

preparados["TIENE_TEXTO_CAUSA"] = (
    textos.notna().any(axis=1).map({True: "SI", False: "NO"})
)

preparados["ESTADO_CODIFICACION"] = (
    preparados["TIENE_CIE_REGISTRADO"].map({
        "SI": "CON_ALGUN_CIE_REGISTRADO",
        "NO": "PENDIENTE_REVISION_SIN_CIE"
    })
)

archivo_base = "sinadef_lima_60mas_2022_2024_preparada.csv"
archivo_revision = "sinadef_lima_60mas_sin_cie_para_revision.csv"

preparados.to_csv(
    archivo_base,
    sep="|",
    encoding="utf-8-sig",
    index=False
)

preparados.loc[
    preparados["TIENE_CIE_REGISTRADO"].eq("NO")
].to_csv(
    archivo_revision,
    sep="|",
    encoding="utf-8-sig",
    index=False
)

# Incluir la base y los resúmenes disponibles, sin añadir el CSV nacional.
archivos_respaldo = [
    archivo_base,
    archivo_revision,
    "calidad_causas_por_anio.csv",
    "revision_descripciones_sin_cie.csv",
    "equivalencias_ubigeo_sinadef_lima.csv",
    "resumen_filtro_temporal.csv",
    "resumen_filtro_geografico.csv",
    "resumen_filtro_edad.csv"
]

with zipfile.ZipFile(
    "SINADEF_Lima_respaldo_preparacion.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as paquete:
    for archivo in archivos_respaldo:
        if Path(archivo).exists():
            paquete.write(archivo)
        else:
            print("Resumen no encontrado:", archivo)

print("Registros conservados:", len(preparados))
print(
    "Pendientes de revisión por ausencia de CIE:",
    preparados["TIENE_CIE_REGISTRADO"].eq("NO").sum()
)

files.download("SINADEF_Lima_respaldo_preparacion.zip")

Registros conservados: 78297
Pendientes de revisión por ausencia de CIE: 14175


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

base = pd.read_csv(
    "sinadef_lima_60mas_2022_2024_preparada.csv",
    sep="|",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

columnas_cie = [
    columna for columna in base.columns
    if "CIE" in columna.upper()
    and columna.upper().startswith("CAUSA")
]

# Normalizar únicamente una copia de los códigos.
codigos_revision = base[columnas_cie].apply(
    lambda columna: (
        columna.str.strip()
        .str.upper()
        .str.replace(".", "", regex=False)
    )
)

# Extraer la categoría de tres caracteres.
# La coincidencia de formato no sustituye la validación del catálogo.
categorias = codigos_revision.apply(
    lambda columna: columna.str.extract(
        r"^([A-Z][0-9]{2})(?:[0-9A-Z]{0,2})$",
        expand=False
    )
)

grupos = {
    "Circulatorias: I00-I99":
        [f"I{i:02d}" for i in range(100)],
    "Isquemicas del corazon: I20-I25":
        [f"I{i}" for i in range(20, 26)],
    "Insuficiencia cardiaca: I50": ["I50"],
    "Insuficiencia renal aguda: N17": ["N17"],
    "Enfermedad renal cronica: N18": ["N18"],
    "Insuficiencia renal no especificada: N19": ["N19"]
}

anios = pd.to_datetime(
    base["FECHA"],
    format="%Y-%m-%d",
    errors="coerce"
).dt.year

resultados = []

for nombre, categorias_grupo in grupos.items():
    # Cada fila cuenta una sola vez dentro de cada grupo.
    aparece = categorias.isin(categorias_grupo).any(axis=1)

    resultados.append({
        "GRUPO": nombre,
        "TOTAL": int(aparece.sum()),
        "2022": int((aparece & anios.eq(2022)).sum()),
        "2023": int((aparece & anios.eq(2023)).sum()),
        "2024_PARCIAL": int((aparece & anios.eq(2024)).sum())
    })

conteo_exploratorio = pd.DataFrame(resultados)
display(conteo_exploratorio)

conteo_exploratorio.to_csv(
    "conteo_exploratorio_menciones_cie.csv",
    encoding="utf-8-sig",
    index=False
)

print("Conteo guardado. No se eliminaron ni recodificaron registros.")

,GRUPO,TOTAL,2022,2023,2024_PARCIAL
0,Circulatorias: I00-I99,23430,11439,8565,3426
1,Isquemicas del corazon: I20-I25,10302,4989,3714,1599
2,Insuficiencia cardiaca: I50,2970,1380,1159,431
3,Insuficiencia renal aguda: N17,1229,659,404,166
4,Enfermedad renal cronica: N18,3070,1407,1241,422
5,Insuficiencia renal no especificada: N19,486,262,140,84


Conteo guardado. No se eliminaron ni recodificaron registros.


In [ ]:
mencion_circulatoria = categorias.isin(
    [f"I{i:02d}" for i in range(100)]
).any(axis=1)

mencion_renal = categorias.isin(
    ["N17", "N18", "N19"]
).any(axis=1)

ambas = mencion_circulatoria & mencion_renal
alguna = mencion_circulatoria | mencion_renal

resumen_coincidencias = pd.DataFrame({
    "CRITERIO": [
        "Mención circulatoria sin mención renal N17-N19",
        "Mención renal N17-N19 sin mención circulatoria",
        "Ambas menciones en el mismo registro",
        "Al menos una de las dos: total sin sumar repetidos"
    ],
    "REGISTROS": [
        int((mencion_circulatoria & ~mencion_renal).sum()),
        int((mencion_renal & ~mencion_circulatoria).sum()),
        int(ambas.sum()),
        int(alguna.sum())
    ]
})

display(resumen_coincidencias)

resumen_coincidencias.to_csv(
    "coincidencias_exploratorias_circulatorias_renales.csv",
    encoding="utf-8-sig",
    index=False
)

print("Se conservan los 78 297 registros de la base.")

,CRITERIO,REGISTROS
0,Mención circulatoria sin mención renal N17-N19,21712
1,Mención renal N17-N19 sin mención circulatoria,3026
2,Ambas menciones en el mismo registro,1718
3,Al menos una de las dos: total sin sumar repet...,26456


Se conservan los 78 297 registros de la base.


In [ ]:
exploratoria = base.loc[alguna].copy()

exploratoria["MENCION_CIRCULATORIA_CIE"] = (
    mencion_circulatoria.loc[alguna]
    .map({True: "SI", False: "NO_DETECTADA"})
)

exploratoria["MENCION_RENAL_N17_N19"] = (
    mencion_renal.loc[alguna]
    .map({True: "SI", False: "NO_DETECTADA"})
)

exploratoria["GRUPO_EXPLORATORIO"] = (
    "RENAL_SIN_MENCION_CIRCULATORIA_DETECTADA"
)

exploratoria.loc[
    mencion_circulatoria.loc[alguna],
    "GRUPO_EXPLORATORIO"
] = "CIRCULATORIA_SIN_MENCION_RENAL_DETECTADA"

exploratoria.loc[
    ambas.loc[alguna],
    "GRUPO_EXPLORATORIO"
] = "AMBAS_MENCIONES"

archivo_exploratorio = (
    "sinadef_lima_60mas_menciones_circulatorias_renales_exploratorio.csv"
)

exploratoria.to_csv(
    archivo_exploratorio,
    sep="|",
    encoding="utf-8-sig",
    index=False
)

print("Registros del subconjunto exploratorio:", len(exploratoria))
print(
    "Registros del subconjunto sin UBIGEO asignado:",
    exploratoria["UBIGEO_INEI"].eq("").sum()
)
print("Archivo generado:", archivo_exploratorio)

Registros del subconjunto exploratorio: 26456
Registros del subconjunto sin UBIGEO asignado: 4
Archivo generado: sinadef_lima_60mas_menciones_circulatorias_renales_exploratorio.csv


In [ ]:
# Detectar filas completamente idénticas.
identicas = base.duplicated(keep=False)

# El primer campo del archivo original es el correlativo Nº.
# Comparar también todas las demás columnas, excluyendo ese campo.
columna_correlativo = base.columns[0]
columnas_comparacion = base.columns[1:].tolist()

coincidencias = base.duplicated(
    subset=columnas_comparacion,
    keep=False
)

resumen_duplicados = pd.DataFrame({
    "REVISION": [
        "Total de registros revisados",
        "Filas completamente idénticas, incluido el correlativo",
        "Filas coincidentes al excluir el correlativo",
        "Coincidencias que pertenecen al subconjunto exploratorio"
    ],
    "REGISTROS": [
        len(base),
        int(identicas.sum()),
        int(coincidencias.sum()),
        int((coincidencias & alguna).sum())
    ]
})

print("Campo excluido en la segunda comparación:", columna_correlativo)
display(resumen_duplicados)

resumen_duplicados.to_csv(
    "resumen_revision_duplicados.csv",
    encoding="utf-8-sig",
    index=False
)

base.loc[coincidencias].to_csv(
    "sinadef_coincidencias_para_revisar.csv",
    sep="|",
    encoding="utf-8-sig",
    index=False
)

print("Revisión guardada. No se eliminó ninguna fila.")

Campo excluido en la segunda comparación: Nº


,REVISION,REGISTROS
0,Total de registros revisados,78297
1,"Filas completamente idénticas, incluido el cor...",0
2,Filas coincidentes al excluir el correlativo,10
3,Coincidencias que pertenecen al subconjunto ex...,4


Revisión guardada. No se eliminó ninguna fila.


In [ ]:
# Preparar una tabla auxiliar sin modificar la base.
trabajo_diario = base[
    ["FECHA", "UBIGEO_INEI", "DISTRITO DOMICILIO"]
].copy()

trabajo_diario["FECHA"] = pd.to_datetime(
    trabajo_diario["FECHA"],
    format="%Y-%m-%d",
    errors="raise"
).dt.strftime("%Y-%m-%d")

trabajo_diario["N_CON_CIE"] = (
    base["TIENE_CIE_REGISTRADO"].eq("SI").astype(int)
)
trabajo_diario["N_SIN_CIE"] = (
    base["TIENE_CIE_REGISTRADO"].eq("NO").astype(int)
)
trabajo_diario["N_MENCION_CIRCULATORIA"] = (
    mencion_circulatoria.astype(int)
)
trabajo_diario["N_MENCION_RENAL_N17_N19"] = (
    mencion_renal.astype(int)
)
trabajo_diario["N_AMBAS_MENCIONES"] = ambas.astype(int)
trabajo_diario["N_ALGUNA_MENCION"] = alguna.astype(int)
trabajo_diario["N_COINCIDENCIAS_PENDIENTES"] = (
    coincidencias.astype(int)
)

# Separar las 11 filas cuya ubicación distrital está pendiente.
con_ubicacion = trabajo_diario["UBIGEO_INEI"].ne("")

base.loc[~con_ubicacion].to_csv(
    "sinadef_registros_sin_ubigeo_para_revision.csv",
    sep="|",
    encoding="utf-8-sig",
    index=False
)

# Agrupar únicamente combinaciones distrito-fecha presentes en la base.
diaria = (
    trabajo_diario.loc[con_ubicacion]
    .groupby(["UBIGEO_INEI", "FECHA"], as_index=False)
    .agg(
        DISTRITO=("DISTRITO DOMICILIO", "first"),
        N_REGISTROS_TOTAL=("FECHA", "size"),
        N_CON_CIE=("N_CON_CIE", "sum"),
        N_SIN_CIE=("N_SIN_CIE", "sum"),
        N_MENCION_CIRCULATORIA=("N_MENCION_CIRCULATORIA", "sum"),
        N_MENCION_RENAL_N17_N19=("N_MENCION_RENAL_N17_N19", "sum"),
        N_AMBAS_MENCIONES=("N_AMBAS_MENCIONES", "sum"),
        N_ALGUNA_MENCION=("N_ALGUNA_MENCION", "sum"),
        N_COINCIDENCIAS_PENDIENTES=(
            "N_COINCIDENCIAS_PENDIENTES", "sum"
        )
    )
    .sort_values(["UBIGEO_INEI", "FECHA"])
)

diaria.to_csv(
    "sinadef_lima_60mas_distrito_dia_exploratorio.csv",
    sep="|",
    encoding="utf-8-sig",
    index=False
)

print("Filas distrito-día generadas:", len(diaria))
print("Registros incluidos:", diaria["N_REGISTROS_TOTAL"].sum())
print("Con alguna mención exploratoria:",
      diaria["N_ALGUNA_MENCION"].sum())
print("Registros apartados por ubicación pendiente:",
      int((~con_ubicacion).sum()))
print("Primera fecha:", diaria["FECHA"].min())
print("Última fecha:", diaria["FECHA"].max())

Filas distrito-día generadas: 25695
Registros incluidos: 78286
Con alguna mención exploratoria: 26452
Registros apartados por ubicación pendiente: 11
Primera fecha: 2022-01-01
Última fecha: 2024-05-05


In [ ]:
fechas = pd.to_datetime(
    diaria["FECHA"],
    format="%Y-%m-%d"
)

calendario = pd.date_range(
    fechas.min(),
    fechas.max(),
    freq="D"
)

# Detectar fechas sin registros en ninguno de los distritos.
fechas_sin_registros = calendario.difference(
    pd.DatetimeIndex(fechas.unique())
)

disponibilidad = (
    diaria.groupby(["UBIGEO_INEI", "DISTRITO"], as_index=False)
    .agg(
        PRIMERA_FECHA=("FECHA", "min"),
        ULTIMA_FECHA=("FECHA", "max"),
        DIAS_CON_FILA=("FECHA", "nunique"),
        REGISTROS=("N_REGISTROS_TOTAL", "sum"),
        SIN_CIE=("N_SIN_CIE", "sum"),
        MENCIONES_EXPLORATORIAS=("N_ALGUNA_MENCION", "sum")
    )
)

disponibilidad["DIAS_SIN_FILA_DENTRO_DEL_CORTE"] = (
    len(calendario) - disponibilidad["DIAS_CON_FILA"]
)

disponibilidad["PORCENTAJE_SIN_CIE"] = (
    100 * disponibilidad["SIN_CIE"] / disponibilidad["REGISTROS"]
).round(2)

disponibilidad.to_csv(
    "disponibilidad_distrital_sinadef.csv",
    encoding="utf-8-sig",
    index=False
)

pd.DataFrame({
    "FECHA": fechas_sin_registros.strftime("%Y-%m-%d")
}).to_csv(
    "fechas_sin_registros_en_toda_lima.csv",
    encoding="utf-8-sig",
    index=False
)

print("Distritos presentes:", len(disponibilidad))
print("Días calendario dentro del corte:", len(calendario))
print("Días con registros en algún distrito:", fechas.nunique())
print("Días sin registros en todos los distritos:",
      len(fechas_sin_registros))

print("\nMuestra: cinco distritos con menos días con filas")
display(
    disponibilidad.sort_values("DIAS_CON_FILA").head(5)
)

Distritos presentes: 43
Días calendario dentro del corte: 856
Días con registros en algún distrito: 856
Días sin registros en todos los distritos: 0

Muestra: cinco distritos con menos días con filas


,UBIGEO_INEI,DISTRITO,PRIMERA_FECHA,ULTIMA_FECHA,DIAS_CON_FILA,REGISTROS,SIN_CIE,MENCIONES_EXPLORATORIAS,DIAS_SIN_FILA_DENTRO_DEL_CORTE,PORCENTAJE_SIN_CIE
37,150138,SANTA MARIA DEL MAR,2022-02-12,2023-12-12,14,14,3,6,842,21.43
28,150129,SAN BARTOLO,2022-01-11,2024-04-28,51,52,12,12,805,23.08
26,150127,PUNTA NEGRA,2022-01-25,2024-04-24,59,59,20,22,797,33.90
25,150126,PUNTA HERMOSA,2022-01-04,2024-04-29,69,70,8,26,787,11.43
23,150124,PUCUSANA,2022-01-24,2024-04-26,76,77,16,18,780,20.78


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

archivos = [
    "sinadef_lima_60mas_2022_2024_preparada.csv",
    "sinadef_lima_60mas_sin_cie_para_revision.csv",
    "sinadef_lima_60mas_menciones_circulatorias_renales_exploratorio.csv",
    "sinadef_lima_60mas_distrito_dia_exploratorio.csv",
    "sinadef_registros_sin_ubigeo_para_revision.csv",
    "sinadef_coincidencias_para_revisar.csv",
    "equivalencias_ubigeo_sinadef_lima.csv",
    "calidad_causas_por_anio.csv",
    "revision_descripciones_sin_cie.csv",
    "conteo_exploratorio_menciones_cie.csv",
    "coincidencias_exploratorias_circulatorias_renales.csv",
    "resumen_revision_duplicados.csv",
    "disponibilidad_distrital_sinadef.csv",
    "fechas_sin_registros_en_toda_lima.csv",
    "resumen_filtro_temporal.csv",
    "resumen_filtro_geografico.csv",
    "resumen_filtro_edad.csv"
]

faltantes = [
    nombre for nombre in archivos
    if not Path(nombre).is_file()
]

if faltantes:
    print("Faltan estos archivos; revisar antes de respaldar:")
    for nombre in faltantes:
        print("-", nombre)
else:
    notas = """SINADEF — Preparación preliminar

Ámbito: domicilio en departamento Lima, provincia Lima.
Edad: 60 años a más, expresada en años.
Filtro temporal solicitado: 2022–2024.
Cobertura encontrada: 2022-01-01 a 2024-05-05.
2024 es parcial.

Base completa del recorte: 78 297 registros.
Sin UBIGEO distrital asignado: 11 registros.
Sin códigos CIE en los seis campos: 14 175 registros.
Estos registros conservan texto en alguna descripción de causa.

Subconjunto exploratorio:
Mención de I00–I99 o N17–N19 en cualquiera de los campos A–F.
26 456 registros; 4 sin UBIGEO asignado.
No constituye una definición final de mortalidad cardiorrenal.
La coexistencia de menciones no confirma un síndrome cardiorrenal.
Las menciones no identifican automáticamente la causa básica.

Revisión de duplicados:
0 filas completamente idénticas.
10 filas coincidentes al excluir el correlativo Nº.
Se conservaron todas; requieren revisión.

Tabla distrito-día:
25 695 filas con 78 286 registros.
Incluye 26 452 registros con alguna mención exploratoria.
No se completaron fechas ausentes con ceros.

Pendiente:
Definir el desenlace y los criterios diagnósticos con Iris.
Revisar registros sin CIE, ubicaciones y coincidencias pendientes.
Evaluar cobertura meteorológica y nivel territorial de análisis.
Conservar el cuaderno de Colab para reproducir el procesamiento.
"""

    paquete = "SINADEF_preparacion_preliminar_respaldo.zip"

    with zipfile.ZipFile(
        paquete, "w", zipfile.ZIP_DEFLATED
    ) as archivo_zip:
        for nombre in archivos:
            archivo_zip.write(nombre)

        archivo_zip.writestr(
            "LEEME_SINADEF_preliminar.txt",
            notas.encode("utf-8")
        )

    print("Respaldo generado con", len(archivos), "CSV y notas.")
    files.download(paquete)

Respaldo generado con 17 CSV y notas.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>